# IGPO｜Information Gain-based Policy Optimization

论文：[arXiv:2510.14967](https://arxiv.org/abs/2510.14967)（ICLR 2026）  
仓库：https://github.com/ankknaiii/igpo-agentic-search

**问题**：小 group 下 easy 全对 / hard 全错 → GRPO advantage=0 → 没梯度  
**IGPO**：每轮 search 后 teacher-force GT，用相邻轮 `P(GT)` 差值作过程奖励

> 先点 **Runtime → Change runtime type → T4 GPU → Save**，再按顺序跑下面格子。

In [ ]:
#@title 1) Bootstrap（自动 git clone，无需上传 zip）
import os, sys, subprocess, zipfile
from pathlib import Path

REPO_URL = "https://github.com/ankknaiii/igpo-agentic-search.git"
ROOT = Path("/content/igpo-agentic-search")

def sh(cmd: str):
    print("$", cmd)
    subprocess.check_call(cmd, shell=True)

# Colab 从 GitHub 打开 notebook 时，工作区只有 ipynb，需要拉取完整仓库
if not (ROOT / "igpo").exists():
    zip_path = Path("/content/igpo-agentic-search.zip")
    if zip_path.exists():
        with zipfile.ZipFile(zip_path) as z:
            z.extractall("/content")
        if not (ROOT / "igpo").exists() and Path("/content/igpo").exists():
            ROOT = Path("/content")
        print("bootstrapped from zip")
    else:
        if ROOT.exists():
            sh(f"rm -rf {ROOT}")
        sh(f"git clone --depth 1 {REPO_URL} {ROOT}")

assert (ROOT / "igpo").exists(), f"repo incomplete under {ROOT}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("cwd=", os.getcwd())
sh("pip install -q -r requirements.txt")
sh("pip install -q -e .")
print("bootstrap ok")

In [ ]:
#@title 2) 单测：GRPO collapse vs IGPO 仍有信号
import subprocess
subprocess.check_call("python scripts/smoke_test.py", shell=True)
subprocess.check_call("pytest -q tests/", shell=True)
print("unit tests passed")

In [ ]:
#@title 3) 迷你训练冒烟（1 step，确认能跑通）
import torch
from igpo.train.trainer import TrainConfig, run_training

print("cuda=", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

smoke_cfg = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    algo="igpo",
    max_steps=1,
    prompts_per_step=1,
    group_size=2,
    max_turns=2,
    max_new_tokens=64,
    learning_rate=1e-5,
    info_gain_type="prob_diff",
    info_gain_norm_mode="separate",
    output_dir="./outputs/smoke",
)
smoke_hist = run_training(smoke_cfg)
print("smoke train ok:", smoke_hist[-1])

In [ ]:
#@title 4) 正式短训 IGPO（可改 max_steps）
from igpo.train.trainer import TrainConfig, run_training

cfg = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    algo="igpo",          # 对照可改 "grpo"
    max_steps=8,
    prompts_per_step=2,
    group_size=4,
    max_turns=3,
    max_new_tokens=128,
    learning_rate=1e-5,
    info_gain_type="prob_diff",
    info_gain_norm_mode="separate",
    output_dir="./outputs/igpo_colab",
)
history = run_training(cfg)
history[-1]

In [ ]:
#@title 5) 曲线：F1 / collapse / |IG|
import matplotlib.pyplot as plt

steps = [h.step for h in history]
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].plot(steps, [h.mean_f1 for h in history]); axes[0].set_title("mean F1")
axes[1].plot(steps, [h.collapse_rate for h in history]); axes[1].set_title("outcome collapse rate")
axes[2].plot(steps, [h.mean_abs_ig for h in history]); axes[2].set_title("mean |IG|")
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout(); plt.show()

### 面试口述
1. **Advantage collapse**：group 内 outcome 全同 → z-score=0  
2. **IG 奖励**：`r_t = P(GT|ctx_t)-P(GT|ctx_{t-1})`，teacher forcing，内生低 cost  
3. **稠密优势**：IG turns + 终局 F1 → separate z-norm → γ 折扣回传  
4. **对比**：相对 MCTS / 外部 RM，更不易 hacking，且每条样本都有梯度信号